# Séance 6 — Agrégation, jointures et modèle en étoile

**Durée pratique : 3 h** &nbsp;·&nbsp; Ateliers 6.1 à 6.4

## Ce que vous saurez faire à la fin

- distinguer `agg`, `transform` et `apply` et choisir le bon ;
- contrôler la cardinalité d'une jointure et diagnostiquer une explosion de lignes ;
- passer du format large au format long et inversement ;
- livrer un modèle en étoile exploitable directement par Power BI ou Tableau.

> **Convention de nommage du module.** Le code est écrit en anglais et suit la PEP 8 :
> fonctions et variables en `snake_case`, constantes en `MAJUSCULES`. Les **noms de colonnes**
> restent en français parce qu'ils viennent de la source : renommer les colonnes d'un fichier
> d'entrée est une transformation comme une autre, elle se décide et se documente, elle ne se
> fait pas par réflexe. Vous rencontrerez cette situation partout en entreprise.

In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

print('Données disponibles :')
for path in sorted(RAW_DIR.glob('*')):
    print(' ', path.name)


# Parquet conserve les types (dates, entiers, catégories) là où le CSV les perd :
# c'est le format à privilégier entre deux étapes d'un pipeline. Repli automatique
# sur le CSV si pyarrow n'est pas installé.
def save_dataset(df, name):
    try:
        path = PROCESSED_DIR / f'{name}.parquet'
        df.to_parquet(path, index=False)
    except ImportError:
        path = PROCESSED_DIR / f'{name}.csv'
        df.to_csv(path, index=False)
        print('(pyarrow absent : repli sur le CSV)')
    print('écrit :', path.name, df.shape)
    return path


def load_dataset(name):
    parquet_path = PROCESSED_DIR / f'{name}.parquet'
    csv_path = PROCESSED_DIR / f'{name}.csv'
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f'{name} introuvable : exécutez le notebook précédent')


def dataset_exists(name):
    return ((PROCESSED_DIR / f'{name}.parquet').exists()
            or (PROCESSED_DIR / f'{name}.csv').exists())

Données disponibles :
  capteurs.csv
  clients.csv
  magasins.csv
  produits.csv
  ventes_brutes.csv
  ventes_extrait.json
  ventes_extrait.xlsx


In [3]:
sales = load_dataset('ventes_transformees')
customers = pd.read_csv(RAW_DIR / 'clients.csv')
products = pd.read_csv(RAW_DIR / 'produits.csv')
stores = pd.read_csv(RAW_DIR / 'magasins.csv')

sales['date_commande'] = pd.to_datetime(sales['date_commande'], errors='coerce')

for label, frame in [('sales', sales), ('customers', customers),
                     ('products', products), ('stores', stores)]:
    print(f'{label:<10} {frame.shape}')

sales      (3446, 22)
customers  (3060, 5)
products   (39, 5)
stores     (9, 5)


---
## Atelier 6.1 — groupby : agg, transform, apply (50 min)

### Partie guidée

In [4]:
# agg : réduit chaque groupe à une ligne
by_category = sales.groupby('categorie').agg(
    nb_commandes=('id_commande', 'count'),
    ca_total=('montant', 'sum'),
    panier_moyen=('montant', 'mean'),
    panier_median=('montant', 'median'),
    quantite_totale=('quantite', 'sum'),
).round(2).sort_values('ca_total', ascending=False)

by_category

,nb_commandes,ca_total,panier_moyen,panier_median,quantite_totale
categorie,,,,,
Informatique,1034,3850629.32,4015.25,1121.20,6843
Audio,830,2350725.57,3044.98,1133.41,3758
Mobilier,765,1790441.40,2511.14,684.34,4385
Accessoire,804,945129.60,1280.66,818.84,2344


Comparez `panier_moyen` et `panier_median`. Un écart important signale une distribution
asymétrique : quelques commandes très élevées tirent la moyenne vers le haut. Publier la
seule moyenne dans un tableau de bord donnerait une image fausse du panier typique.

In [5]:
# transform : renvoie un résultat ALIGNÉ sur les lignes d'origine
sales['ca_categorie'] = sales.groupby('categorie')['montant'].transform('sum')
sales['part_dans_categorie'] = (sales['montant'] / sales['ca_categorie'] * 100).round(4)

print(sales[['categorie', 'montant', 'ca_categorie', 'part_dans_categorie']].head())
print('\nSomme des parts par catégorie :')
print(sales.groupby('categorie')['part_dans_categorie'].sum().round(2))

      categorie   montant  ca_categorie  part_dans_categorie
0  Informatique   457.881  3.850629e+06               0.0119
1         Audio  1419.040  2.350726e+06               0.0604
2    Accessoire  2570.700  9.451296e+05               0.2720
3         Audio  2107.980  2.350726e+06               0.0897
4         Audio  1445.300  2.350726e+06               0.0615

Somme des parts par catégorie :
categorie
Accessoire      100.0
Audio           100.0
Informatique    100.0
Mobilier        100.0
Name: part_dans_categorie, dtype: float64


**La règle est simple.** `agg` réduit (une ligne par groupe), `transform` conserve
(autant de lignes qu'à l'entrée). Chaque fois que vous voulez comparer une valeur à la
statistique de son groupe, c'est `transform` qu'il vous faut, jamais un `merge` sur un
résultat de `agg`.

In [6]:
# apply : le plus souple, mais le plus lent. À réserver aux cas irréductibles.
def summarize_group(group):
    return pd.Series({
        'nb': len(group),
        'top_produit': group['id_produit'].mode().iloc[0] if len(group) else None,
        'part_annulees': (group['statut'] == 'annule').mean().round(3),
    })


sales.groupby('canal').apply(summarize_group, include_groups=False)

,nb,top_produit,part_annulees
canal,,,
boutique,1158,P0001,0.161
telephone,614,P0039,0.169
web,1674,P0020,0.165


### Partie autonome

In [7]:
# Q1. Construire un tableau croisé : chiffre d'affaires par catégorie et par canal,
ca_categorie_canal = sales.pivot_table(
    index='categorie',
    columns='canal',
    values='montant',
    aggfunc='sum',
    margins=True,
    margins_name='Total',
).round(0)

ca_categorie_canal

canal,boutique,telephone,web,Total
categorie,,,,
Accessoire,400308.0,142417.0,402405.0,945130.0
Audio,440161.0,197020.0,1713545.0,2350726.0
Informatique,1821806.0,329173.0,1699650.0,3850629.0
Mobilier,878332.0,595819.0,316291.0,1790441.0
Total,3540606.0,1264429.0,4131891.0,8936926.0


In [8]:
# Q2. Pour chaque client, calculer : nombre de commandes, CA total, panier moyen,
#     date de première et de dernière commande, nombre de catégories distinctes achetées.
#     Puis identifier les dix meilleurs clients par CA.
customer_profile = sales.groupby('id_client').agg(
    nb_commandes=('id_commande', 'count'),
    ca_total=('montant', 'sum'),
    panier_moyen=('montant', 'mean'),
    premiere_commande=('date_commande', 'min'),
    derniere_commande=('date_commande', 'max'),
    nb_categories=('categorie', 'nunique'),
).round(2)

# Dix meilleurs clients par CA
top_10_customers = customer_profile.nlargest(10, 'ca_total')
top_10_customers

,nb_commandes,ca_total,panier_moyen,premiere_commande,derniere_commande,nb_categories
id_client,,,,,,
C02327,1,903506.12,903506.12,2024-12-28 14:36:00,2024-12-28 14:36:00,1
C00685,2,626429.42,313214.71,2023-09-07 16:17:00,2024-03-06 15:22:00,2
990028,1,618310.51,618310.51,2023-02-15 18:01:00,2023-02-15 18:01:00,1
C00553,1,615738.98,615738.98,2023-04-11 15:16:00,2023-04-11 15:16:00,1
C00309,2,599010.14,299505.07,2023-11-04 13:19:00,2024-04-01 13:33:00,2
C01626,4,442908.98,110727.25,2023-01-16 13:14:00,2023-12-22 15:36:00,3
C02431,2,384724.84,192362.42,2023-04-20 18:28:00,2024-05-19 08:57:00,1
C01512,2,319718.75,159859.38,2024-03-29 08:40:00,2024-08-29 12:47:00,2
C00488,1,154562.28,154562.28,2024-09-14 18:04:00,2024-09-14 18:04:00,1


In [9]:
# Q3. Avec transform, ajouter une colonne signalant les commandes dont le montant
#     dépasse de plus de 50 % le montant médian de leur catégorie.
#     Quelle part des commandes cela représente-t-il ?

sales['mediane_categorie'] = sales.groupby('categorie')['montant'].transform('median')
sales['est_montant_eleve'] = sales['montant'] > 1.5 * sales['mediane_categorie']

part_elevees = sales['est_montant_eleve'].mean()
print(f"{sales['est_montant_eleve'].sum()} commandes concernées ({part_elevees:.2%})")

1057 commandes concernées (30.67%)


---
## Atelier 6.2 — Jointures et explosion de lignes (55 min)

### Partie guidée

In [10]:
print('Lignes avant jointure :', len(sales))

with_products = sales.merge(products, on='id_produit', how='left', suffixes=('', '_prod'))
print('Après jointure produits :', len(with_products))

n_orphans = with_products['libelle'].isna().sum()
print(f'Lignes sans produit correspondant : {n_orphans}',
      f'({n_orphans / len(with_products):.2%})')

unknown_ids = sales.loc[~sales['id_produit'].isin(products['id_produit']), 'id_produit']
print('Identifiants fautifs :', unknown_ids.unique())

Lignes avant jointure : 3446
Après jointure produits : 3446
Lignes sans produit correspondant : 13 (0.38%)
Identifiants fautifs : <ArrowStringArray>
['P9999']
Length: 1, dtype: str


**Le réflexe à acquérir : compter les lignes avant et après chaque jointure.** Un `left join`
conserve le nombre de lignes *seulement si* la clé est unique à droite. Si elle ne l'est pas,
le résultat se multiplie sans avertissement.

In [11]:
# La table clients est saine : sa clé primaire est bien unique.
print('Identifiants clients dupliqués :', customers['id_client'].duplicated().sum())

# On simule donc le cas d'une table de droite dégradée — situation banale dès qu'une
# extraction est relancée deux fois ou qu'un export concatène deux périodes.
customers_with_duplicates = pd.concat([customers, customers.head(50)], ignore_index=True)
exploded = sales.merge(customers_with_duplicates, on='id_client', how='left')
print(f'{len(sales)} lignes -> {len(exploded)} après jointure sur une clé non unique')
print('Aucune erreur levée. Le problème ne se verra que bien plus loin.')

Identifiants clients dupliqués : 0
3446 lignes -> 3510 après jointure sur une clé non unique
Aucune erreur levée. Le problème ne se verra que bien plus loin.


In [12]:
# Le paramètre validate transforme le problème silencieux en erreur explicite
try:
    sales.merge(customers_with_duplicates, on='id_client', how='left', validate='many_to_one')
except Exception as error:
    print(type(error).__name__, ':', error)

MergeError : Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 id_client
   C00001
   C00002
   C00003
   C00004
   C00005 ...


Prenez l'habitude de passer `validate` sur **toutes** vos jointures. Le coût est nul et
l'erreur qu'il attrape est parmi les plus coûteuses à diagnostiquer plus tard.

### Partie autonome

In [13]:
# Q4. Assembler les quatre tables en un seul DataFrame analytique :
#     sales + products + customers + stores.
#     Contraintes : vérifier le nombre de lignes après chaque étape, utiliser validate,
#     et traiter explicitement les clés orphelines (documentez votre choix).

row_counts = {'sales (départ)': len(sales)}


def join_step(left, right, key, validate, name, suffix):
    """Jointure left avec validate, indicateur d'orphelins et comptage des lignes."""
    merged = left.merge(right, on=key, how='left', validate=validate,
                        suffixes=('', suffix), indicator=f'_merge_{name}')
    merged[f'orphelin_{name}'] = merged[f'_merge_{name}'].eq('left_only')
    merged = merged.drop(columns=f'_merge_{name}')

    n_orphans = merged[f'orphelin_{name}'].sum()
    row_counts[f'après {name}'] = len(merged)
    print(f'{name:<10} {len(left)} -> {len(merged)} lignes | '
          f'{n_orphans} orphelines ({n_orphans / len(merged):.2%})')
    assert len(merged) == len(left), f'Explosion ou perte de lignes sur {name}'
    return merged


analytic = sales.copy()
analytic = join_step(analytic, products,  'id_produit', 'many_to_one', 'produit', '_prod')
analytic = join_step(analytic, customers, 'id_client',  'many_to_one', 'client',  '_client')
analytic = join_step(analytic, stores,    'id_magasin', 'many_to_one', 'magasin', '_magasin')

print()
print(pd.Series(row_counts, name='nb_lignes'))
print('Colonnes :', analytic.shape[1])

produit    3446 -> 3446 lignes | 13 orphelines (0.38%)
client     3446 -> 3446 lignes | 0 orphelines (0.00%)
magasin    3446 -> 3446 lignes | 0 orphelines (0.00%)

sales (départ)    3446
après produit     3446
après client      3446
après magasin     3446
Name: nb_lignes, dtype: int64
Colonnes : 41


In [14]:
# Q5. Écrire une fonction safe_merge(left, right, **kwargs) qui effectue la jointure
#     et lève une exception si le nombre de lignes change alors qu'un 'left join'
#     était demandé. Elle doit aussi signaler le taux de clés non appariées.

def safe_merge(left, right, **kwargs):
    ...  # TODO

---
## Atelier 6.3 — Formats long et large (35 min)

In [15]:
# Format large : une colonne par mois, lisible par un humain
wide = sales.pivot_table(index='categorie',
                         columns=sales['date_commande'].dt.month,
                         values='montant', aggfunc='sum').round(0)
wide.columns = [f'mois_{month:02d}' for month in wide.columns]
wide.head()

,mois_01,mois_02,mois_03,mois_04,mois_05,mois_06,mois_07,mois_08,mois_09,mois_10,mois_11,mois_12
categorie,,,,,,,,,,,,
Accessoire,70140.0,69825.0,78718.0,64086.0,53144.0,56204.0,63277.0,35603.0,204415.0,74118.0,73800.0,101798.0
Audio,70417.0,78089.0,70825.0,702133.0,72004.0,77654.0,211856.0,70017.0,86854.0,81372.0,729655.0,99850.0
Informatique,633996.0,743869.0,219247.0,97221.0,91704.0,136936.0,114452.0,405540.0,99473.0,121727.0,139858.0,1046605.0
Mobilier,67588.0,58925.0,54101.0,60708.0,543210.0,53594.0,63604.0,38109.0,673721.0,47360.0,77083.0,52438.0


In [16]:
# Format long : une ligne par observation, exploitable par une machine
long = wide.reset_index().melt(id_vars='categorie', var_name='mois', value_name='ca')
long.head()

,categorie,mois,ca
0,Accessoire,mois_01,70140.0
1,Audio,mois_01,70417.0
2,Informatique,mois_01,633996.0
3,Mobilier,mois_01,67588.0
4,Accessoire,mois_02,69825.0


**Lequel choisir ?** Le format long pour tout traitement programmatique et pour les outils
de BI, qui attendent une ligne par fait. Le format large uniquement pour l'affichage final
destiné à un lecteur humain.

In [17]:
# Q6. Repasser du format long au format large sans utiliser pivot_table,
#     puis vérifier que vous retrouvez exactement le tableau de départ.

# TODO

---
## Atelier 6.4 — Livrer un modèle en étoile (40 min)

Un outil de BI attend une **table de faits** (une ligne par événement mesurable, contenant
des clés étrangères et des mesures numériques) entourée de **tables de dimensions** (une ligne
par entité, contenant ses attributs descriptifs). Livrer une seule table plate oblige
l'utilisateur à refaire ce travail dans l'outil, souvent moins bien.

In [18]:
# Dimension temps : indispensable, souvent oubliée
full_range = pd.date_range(sales['date_commande'].min().normalize(),
                           sales['date_commande'].max().normalize(), freq='D')

dim_date = pd.DataFrame({'date': full_range})
dim_date['cle_date'] = dim_date['date'].dt.strftime('%Y%m%d').astype(int)
dim_date['annee'] = dim_date['date'].dt.year
dim_date['mois'] = dim_date['date'].dt.month
dim_date['nom_mois'] = dim_date['date'].dt.month_name()
dim_date['trimestre'] = dim_date['date'].dt.quarter
dim_date['semaine_iso'] = dim_date['date'].dt.isocalendar().week.astype(int)
dim_date['jour_semaine'] = dim_date['date'].dt.dayofweek
dim_date['nom_jour'] = dim_date['date'].dt.day_name()
dim_date['est_weekend'] = dim_date['jour_semaine'] >= 5

print(dim_date.shape)
dim_date.head(3)

(730, 10)


,date,cle_date,annee,mois,nom_mois,trimestre,semaine_iso,jour_semaine,nom_jour,est_weekend
0,2023-01-01,20230101,2023,1,January,1,52,6,Sunday,True
1,2023-01-02,20230102,2023,1,January,1,1,0,Monday,False
2,2023-01-03,20230103,2023,1,January,1,1,1,Tuesday,False


La dimension temps couvre **toute la plage en continu**, y compris les jours sans aucune
vente. Sans elle, un graphique d'évolution saute les jours vides et donne une impression
de continuité trompeuse.

In [19]:
# Q7. Construire le modèle complet et l'exporter en Parquet dans data/processed/star/ :
#     - fact_sales : cle_date, id_client, id_produit, id_magasin, quantite,
#       prix_unitaire, remise_pct, montant, montant_net, statut
#     - dim_date (déjà construite)
#     - dim_customer, dim_product, dim_store : dédoublonnées sur leur clé primaire
#     Vérifiez ensuite l'intégrité référentielle : toute clé de la table de faits
#     doit exister dans la dimension correspondante.

# TODO

In [20]:
# Q8. Écrire un contrôle automatique de l'intégrité référentielle :
#     pour chaque couple (table de faits, dimension), afficher le nombre de clés orphelines.
#     Le contrôle doit échouer bruyamment s'il en trouve.

def check_referential_integrity(fact_table, dimensions):
    """dimensions : dict {nom_colonne_cle: dataframe_dimension}"""
    ...  # TODO

---
## Livrable de la séance

- `data/processed/star/` contenant la table de faits et les quatre dimensions en Parquet ;
- les fonctions `safe_merge` et `check_referential_integrity`, dans `src/` ;
- un schéma (même dessiné à la main et photographié) du modèle en étoile obtenu ;
- le compte des lignes à chaque étape de l'assemblage, avec explication des écarts.

**Ce livrable est le cœur de votre projet.** C'est le fichier que vous ouvrirez dans
Power BI ou Tableau, et celui dont la structure sera discutée en soutenance.